# 02 · GSE65391 · RNA_array · normalise the arrays

Reads the two non-normalized files and the series matrix (notebook 00), `GPL10558.soft.gz`, and
`metadata.rds` (notebook 01). Writes `data/run_artifacts/GSE65391/probes.rds`.

**Why the non-normalized files.** The values in the GEO series matrix were processed by the authors with
Illumina GenomeStudio ("average" normalization) and then adjusted for batch with the technical replicates
(GEO data-processing note). The non-normalized files hold, for every probe and array, the intensity before
normalization and its detection p-value, so each step can be done here and stated.

**Steps, following best practice for Illumina BeadChips:**
1. **Normalize with `neqc`** (Shi W, Oshlack A, Smyth GK. Optimizing the noise versus bias trade-off for
   Illumina whole genome expression BeadChips. *Nucleic Acids Res* 2010;38:e204). `neqc` estimates the
   background of each array, subtracts it with a normal-exponential model, adds an offset of 16, takes
   log2, and applies quantile normalization so every array has the same distribution. The background is
   estimated from the detection p-values, because these files have no control probes.
2. **Probe quality.** Each probe is rated Perfect, Good, Bad or No match by re-mapping its sequence to the
   genome (Barbosa-Morais NL et al. A re-annotation pipeline for Illumina BeadArrays: improving the
   interpretation of gene expression data. *Nucleic Acids Res* 2010;38:e17; Bioconductor package
   `illuminaHumanv4.db`). Only Perfect and Good probes are used for genes.
3. **Detection.** A probe is **detected** when its detection p-value is below 0.05 in at least 10% of the
   996 arrays.

The batch correction is the next notebook (03), and the collapse from probes to genes the one after (03b).

## Load the packages, the sample table and the array barcodes

In [1]:
source("../src/paths.R")
suppressMessages({library(data.table); library(limma); library(GEOquery); library(Biobase); library(illuminaHumanv4.db)})
meta <- readRDS(art("GSE65391", "metadata.rds"))
pd <- pData(suppressMessages(getGEO(filename = raw("GSE65391", "GSE65391_series_matrix.txt.gz"), getGPL = FALSE)))
barcode <- setNames(sub("Sample name: ", "", pd$description), rownames(pd))
head(barcode, 3)

GSM1594219     GSM1594220     GSM1594221 
"5552251039_D" "5552251041_I" "5552251047_D"

**Explanation**
- `library(data.table)` reads large text files quickly with `fread()`. `library(limma)` provides `neqc()`
  (Ritchie ME et al. limma powers differential expression analyses for RNA-sequencing and microarray
  studies. *Nucleic Acids Res* 2015;43:e47). `library(illuminaHumanv4.db)` holds the probe annotation of the
  array (Dunning M, Lynch A, Eldridge M. illuminaHumanv4.db: Illumina HumanHT12v4 annotation data. R package
  version 1.26.0, Bioconductor).
- The non-normalized files name each array by its barcode, for example `5552251039_D`, not by its GSM
  identifier. The series matrix gives both: each sample's description reads `Sample name: <barcode>`.
- `sub("Sample name: ", "", pd$description)` removes the words and keeps the barcode.
- `setNames(barcodes, GSM identifiers)` names each barcode by its GSM identifier. For example,
  `barcode["GSM1594219"]` is `"5552251039_D"`.

## Read the two non-normalized files

In [2]:
read_nonnorm <- function(file) {
  x <- fread(file, data.table = FALSE)
  int <- seq(2, ncol(x), 2); det <- int + 1
  stopifnot(all(grepl("Detection", names(x)[det])))
  gsm <- names(barcode)[match(names(x)[int], barcode)]
  I <- as.matrix(x[, int]); P <- as.matrix(x[, det])
  dimnames(I) <- dimnames(P) <- list(x$ID_REF, gsm)
  list(I = I, P = P)
}
r1 <- read_nonnorm(raw("GSE65391", "GSE65391_non-normalized_data_Illumina_HT12_V4_R1.txt.gz"))
r2 <- read_nonnorm(raw("GSE65391", "GSE65391_non-normalized_data_Illumina_HT12_V4_R2.txt.gz"))
common <- intersect(rownames(r1$I), rownames(r2$I))
I <- cbind(r1$I[common, ], r2$I[common, ])[, rownames(meta)]
P <- cbind(r1$P[common, ], r2$P[common, ])[, rownames(meta)]
c(arrays_R1 = ncol(r1$I), arrays_R2 = ncol(r2$I), probes_R1 = nrow(r1$I), probes_R2 = nrow(r2$I),
  probes_in_both = length(common), arrays = ncol(I))
table(file = rep(c("R1", "R2"), c(ncol(r1$I), ncol(r2$I))), batch = meta[c(colnames(r1$I), colnames(r2$I)), "batch"])

arrays_R1      arrays_R2      probes_R1      probes_R2 probes_in_both 
           838            158          47231          47323          47231 
        arrays 
           996

    batch
file   1   2
  R1 838   0
  R2   0 158

**Explanation**
- In each file the first column, `ID_REF`, is the probe. After it the columns alternate: the intensity of
  one array, then its detection p-value. For example, column 2 is array `5552269045_B` and column 3 its
  detection p-value.
- `read_nonnorm(file)` reads one file and returns two matrices with the same rows and columns:
  - `seq(2, ncol(x), 2)` gives the intensity columns 2, 4, 6 and so on; `det` the detection columns 3, 5, 7.
  - `stopifnot(...)` stops if a detection column is not where expected.
  - `match(names(x)[int], barcode)` finds each array's barcode in the list, and `names(barcode)[...]` gives
    its GSM identifier.
  - `I` holds the intensities, `P` the detection p-values; `dimnames()` names rows by probe and columns by
    GSM identifier.
- `intersect()` keeps the probes present in both files. Every probe of R1 is in R2; R2 has 92 more, which
  are left out.
- `cbind()` puts the two files side by side; `[, rownames(meta)]` orders the arrays as in the sample table.
- The last table shows that file R1 holds batch 1 and file R2 holds batch 2.

## Normalize with neqc

In [3]:
E <- neqc(I, detection.p = P)
c(min = round(min(E), 2), median = round(median(E), 2), max = round(max(E), 2))

min median    max 
  4.15   4.50  13.47

**Explanation**
- `neqc(I, detection.p = P)` returns the normalized log2 values, probes by arrays, in the same order.
  - It estimates each array's background from the probes that are not detected, using the detection
    p-values, and removes it with a normal-exponential model, so no value becomes negative.
  - It adds 16 before taking log2, which limits the noise of very low values.
  - It then applies quantile normalization: after it, every array has exactly the same distribution of
    values.
- The last line prints the smallest, median and largest normalized value.

## Probe quality and detection

In [4]:
quality <- unlist(as.list(illuminaHumanv4PROBEQUALITY))[rownames(E)]
quality_ok <- !is.na(quality) & grepl("^(Perfect|Good)", quality)
detected <- rowMeans(P < 0.05) >= 0.10
table(quality = sub("[*]+$", "", quality), useNA = "ifany")
c(probes = nrow(E), quality_ok = sum(quality_ok), detected = sum(detected), both = sum(quality_ok & detected))

quality
     Bad     Good No match  Perfect 
   12379     1803      376    32673 

probes quality_ok   detected       both 
     47231      34476      23717      18492

**Explanation**
- `illuminaHumanv4PROBEQUALITY` maps each probe to its quality rating. `as.list()` and `unlist()` turn it
  into a named vector; `[rownames(E)]` puts it in the order of our probes.
- Some ratings carry asterisks, for example `Perfect***`, which give more detail on the match;
  `sub("[*]+$", "", quality)` removes them for the table.
- `quality_ok` is TRUE for Perfect and Good probes. `grepl("^(Perfect|Good)", quality)` is TRUE where the
  rating starts with either word.
- `P < 0.05` is TRUE where a probe is detected on an array; `rowMeans()` gives the share of arrays, and
  `>= 0.10` requires at least 10%. For example, a probe detected on 150 of 996 arrays (15%) is detected.

## Add the gene identifiers and save

**Result.** Of 47,231 probes, 34,476 are rated Perfect or Good, 23,717 are detected, and 18,492 are both.

In [5]:
gpl <- Table(suppressMessages(getGEO(filename = raw("GSE65391", "GPL10558.soft.gz"))))
entrez <- gpl$Entrez_Gene_ID[match(rownames(E), gpl$ID)]
entrez[!is.na(entrez) & !nzchar(entrez)] <- NA
c(probes_with_entrez = sum(!is.na(entrez)))
saveRDS(list(E = E, detected = detected, quality_ok = quality_ok, entrez = entrez), art("GSE65391", "probes.rds"))

probes_with_entrez 
             43960

**Explanation**
- `Table()` returns the platform annotation: one row per probe, with its Entrez gene identifier.
- `match(rownames(E), gpl$ID)` finds each probe's row, and `gpl$Entrez_Gene_ID[...]` its gene identifier.
  Control probes have none; empty identifiers are set to missing.
- `probes.rds` keeps the normalized values of all probes, and for each probe: detected, quality and Entrez
  identifier. Notebook 03 corrects the batch; notebook 03b collapses to genes.